# Seto Tiny 1: pretraining

Цель: получить стабильный базовый pretrain для модели `tiny` (~200M параметров), затем отдельно сделать SFT.

Режимы:
- `smoke` по умолчанию: небольшой корпус и 3 000 шагов для проверки пайплайна;
- `full`: 50 000 шагов на расширенном русском корпусе.

Особенности:
- возобновление с последнего чекпойнта;
- переиспользование уже подготовленных данных;
- русский FineWeb2 как основной источник;
- английский, украинский, Wikipedia и code отключены;
- FP16 для T4.

In [ ]:
!pip install -q tokenizers datasets

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle').exists() else Path('/content')
RUN_MODE = 'smoke'  # 'smoke' для проверки, 'full' для основного запуска
GPU_COUNT = max(1, __import__('torch').cuda.device_count())

if RUN_MODE == 'smoke':
    MAX_SAMPLES_RU = 20_000
    MAX_STEPS = 3_000
else:
    MAX_SAMPLES_RU = 400_000
    MAX_STEPS = 50_000

MAX_SAMPLES_EN = 0
MAX_SAMPLES_UK = 0
MAX_SAMPLES_WIKI = 0
MAX_SAMPLES_CODE = 0
SAVE_EVERY = 500

REPO = WORK / 'seto'
HF_CACHE = WORK / 'seto-hf-cache'
TOKENIZER = WORK / 'seto-tiny-tokenizer'
DATA = WORK / f'seto-tiny-data-{RUN_MODE}'
OUTPUT = WORK / f'seto-tiny-1-{RUN_MODE}'
DATA_READY = DATA / '.ready'

os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')
os.environ['HF_HUB_DISABLE_XET'] = '1'


def run(*args, cwd=None):
    print('+', ' '.join(map(str, args)), flush=True)
    subprocess.run([str(arg) for arg in args], cwd=cwd, check=True)

print(f'Run mode: {RUN_MODE}')
print(f'GPUs: {GPU_COUNT}')
print(f'Max steps: {MAX_STEPS}')
print(f'Output: {OUTPUT}')

In [ ]:
if REPO.exists():
    run('git', 'pull', '--ff-only', cwd=REPO)
else:
    run('git', 'clone', 'https://github.com/dustincorder/seto.git', REPO)

sys.path.insert(0, str(REPO))
print('Repository ready:', REPO)

In [ ]:
import numpy as np


def shard_token_count():
    return sum(path.stat().st_size // np.dtype(np.uint16).itemsize
               for path in sorted((DATA / 'shards').glob('*.bin')))


minimum_tokens = max(1_000_000, MAX_SAMPLES_RU * 300)
existing_tokens = shard_token_count()

if existing_tokens >= minimum_tokens and (TOKENIZER / 'tokenizer.json').exists():
    DATA_READY.touch()
    print(
        f'Using existing corpus: {existing_tokens:,} tokens in {DATA / "shards"}. '
        'Skipping dataset download.'
    )
elif not DATA_READY.exists():
    command = [
        sys.executable, '-u', 'scripts/prepare_data.py',
        '--output-dir', DATA,
        '--tokenizer-dir', TOKENIZER,
        '--max-samples-ru', MAX_SAMPLES_RU,
        '--max-samples-en', MAX_SAMPLES_EN,
        '--max-samples-uk', MAX_SAMPLES_UK,
        '--max-samples-wiki', MAX_SAMPLES_WIKI,
        '--max-samples-technical', MAX_SAMPLES_CODE,
        '--shard-size', '100000000',
    ]
    if (TOKENIZER / 'tokenizer.json').exists():
        command.append('--skip-tokenizer')

    try:
        run(*command, cwd=REPO)
    except subprocess.CalledProcessError as error:
        recovered_tokens = shard_token_count()
        if recovered_tokens < minimum_tokens:
            raise RuntimeError(
                f'Data preparation failed and only {recovered_tokens:,} tokens '
                f'were recovered; expected at least {minimum_tokens:,}.'
            ) from error
        print(
            f'FineWeb2 process exited after writing {recovered_tokens:,} tokens; '
            'using the recovered shards.'
        )

    DATA_READY.touch()
else:
    print('Reusing prepared data:', DATA)

shards = sorted((DATA / 'shards').glob('*.bin'))
if not shards:
    raise RuntimeError(f'No shard files found in {DATA / "shards"}')

total_tokens = shard_token_count()
print(f'Shards: {len(shards)}')
print(f'Tokens: {total_tokens:,} (~{total_tokens / 1e9:.2f}B)')
print('Tokenizer:', TOKENIZER)

In [ ]:
# Запуск или продолжение pretrain.
# train.py сам продолжает с последнего чекпойнта в OUTPUT, если он существует.
# Не используем --init-from: это новый базовый pretrain с нуля.
train_command = [
    'torchrun', '--standalone', '--nproc_per_node=' + str(GPU_COUNT),
    'scripts/train.py',
    '--stage', 'pretrain',
    '--model-config', 'tiny',
    '--data-dir', DATA / 'shards',
    '--output-dir', OUTPUT,
    '--tokenizer', TOKENIZER,
    '--batch-size', '2',
    '--grad-accum', '8',
    '--seq-len', '1024',
    '--lr', '1e-4',
    '--warmup-steps', '500',
    '--max-steps', MAX_STEPS,
    '--save-every', SAVE_EVERY,
    '--log-every', '10',
    '--fp16',
]

print('Training command:')
print(' '.join(map(str, train_command)))
run(*train_command, cwd=REPO)

In [ ]:
# Проверка результата и свободного места после завершения полного запуска.
final_zip = OUTPUT / 'final_pretrain.zip'
checkpoints = sorted((OUTPUT / 'checkpoints_pretrain').glob('seto_step_*.zip'))
print('Latest checkpoint:', checkpoints[-1] if checkpoints else 'none')
print('Final export:', final_zip if final_zip.exists() else 'not created yet')
usage = shutil.disk_usage(WORK)
print(f'Free disk: {usage.free / 1024**3:.2f} GiB')

## После pretrain

Не оценивай эту модель через chat-template как готового ассистента. Сначала проверь raw continuation на нескольких фиксированных началах текста, затем запускай SFT.

Если Kaggle остановит сессию, повторно запусти ячейки сверху: данные и последний чекпойнт будут переиспользованы. Не удаляй `/kaggle/working/seto-tiny-1` между сессиями.